# Brandenburg-Höhenkacheln nach R2 (bDOM)

Prozessiert das **bildbasierte DOM (bDOM)** von Brandenburg flächendeckend zu den
binären Höhen-Kacheln der Elevation API und lädt sie nach Cloudflare R2.

**Warum bDOM statt ALS-LAZ?** Gleiche DOM-Semantik (Oberfläche inkl. Bäume/Gebäude),
aber ~5× kleiner (~22 MB statt ~114 MB pro Kachel) und ohne Punktwolken-Gridding.

**Auflösung:** bDOM ist nativ **0,2 m** (5000×5000 pro km-Kachel). Wir rechnen auf
**1 m** herunter (`Resampling.max` = höchster Punkt gewinnt, wie das alte DSM), damit das
Format byte-kompatibel bleibt und der Worker unverändert läuft. (Feiner ginge, würde aber
Tile-Größe/R2-Speicher vervielfachen und Worker-Anpassungen erfordern.)

**Format (byte-kompatibel zu den bestehenden Kacheln):** 1000×1000, 1 m, Uint16 in cm,
Little-Endian, Zeile 0 = Süden, Index `row*1000+col`, nodata = 0.

**Ablauf:** alle bDOM-Kacheln auflisten → bereits auf R2 vorhandene überspringen (Resume)
→ je Kachel: ZIP laden → GeoTIFF auf das km-Raster resampeln → spiegeln → `.bin`(+`.bin.gz`)
→ nach R2 hochladen.

**Maßstab ganz BB:** ~30k Kacheln, ~680 GB Download, ~60 GB `.bin` auf R2. Free-Colab
trennt nach ~12 h — einfach die Run-Zelle erneut starten (Resume überspringt Fertiges).

## Vorbereitung (einmalig)
1. Cloudflare → R2 → **Manage R2 API Tokens** → *Create API Token* (Object Read & Write,
   Bucket `windrad-tiles`). Du erhältst **Access Key ID** + **Secret Access Key**.
2. In Colab: 🔑 **Secrets** (linke Leiste) anlegen:
   `R2_ACCOUNT_ID`, `R2_ACCESS_KEY_ID`, `R2_SECRET_ACCESS_KEY` (Notebook-Zugriff aktivieren).
   `R2_ACCOUNT_ID` ist die Account-ID (Cloudflare-Dashboard-URL).

In [ ]:
!pip -q install rasterio boto3 requests tqdm numpy

In [ ]:
import os, io, re, zipfile, gzip, requests, numpy as np, rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from concurrent.futures import ThreadPoolExecutor, as_completed
import boto3
from botocore.config import Config

# --- R2-Zugang: bevorzugt aus Colab-Secrets, sonst aus Umgebungsvariablen ---
try:
    from google.colab import userdata
    def get(k):
        try: return userdata.get(k)
        except Exception: return ''
except Exception:
    def get(k): return os.environ.get(k, '')

R2_ACCOUNT_ID = get('R2_ACCOUNT_ID') or '975505fa80cf3d0f8e0c3b049e9c6112'
R2_ACCESS_KEY = get('R2_ACCESS_KEY_ID')
R2_SECRET_KEY = get('R2_SECRET_ACCESS_KEY')

BUCKET     = 'windrad-tiles'
BDOM_BASE  = 'https://data.geobasis-bb.de/geobasis/daten/bdom/tif'
TILE_SIZE  = 1000      # Meter pro Kachel
GRID       = 1000      # Zellen pro Kante -> 1 m Auflösung
UPLOAD_GZ  = True      # zusätzlich .bin.gz hochladen
WORKERS    = 8         # parallele Downloads/Uploads
# Optionaler Bounding-Box-Filter (UTM33-km): None = ganz BB. Beispiel: (xmin,xmax,ymin,ymax)
BBOX_KM    = None      # z.B. (450, 470, 5720, 5740)

s3 = boto3.client('s3',
    endpoint_url=f'https://{R2_ACCOUNT_ID}.r2.cloudflarestorage.com',
    aws_access_key_id=R2_ACCESS_KEY, aws_secret_access_key=R2_SECRET_KEY,
    region_name='auto',
    config=Config(retries={'max_attempts': 5, 'mode': 'standard'}))
assert R2_ACCESS_KEY and R2_SECRET_KEY, 'R2-Secrets fehlen (siehe Markdown oben).'
print('R2-Client bereit. Account:', R2_ACCOUNT_ID[:8] + '…')

In [ ]:
# Alle existierenden bDOM-Kacheln vom Geoportal auflisten (Dateiname = km-Raster)
def list_all_bdom_tiles():
    html = requests.get(BDOM_BASE + '/', timeout=120).text
    tiles = {(int(x), int(y)) for x, y in re.findall(r'bdom_33(\d+)-(\d+)\.zip', html)}
    return sorted(tiles)

ALL = list_all_bdom_tiles()
if BBOX_KM:
    x0, x1, y0, y1 = BBOX_KM
    ALL = [(x, y) for (x, y) in ALL if x0 <= x <= x1 and y0 <= y <= y1]
print('bDOM-Kacheln im Umfang:', len(ALL))

In [ ]:
# Resume: bereits als .bin auf R2 vorhandene Kacheln ermitteln
def list_done():
    done, token = set(), None
    while True:
        kw = {'Bucket': BUCKET, 'Prefix': 'tile_'}
        if token: kw['ContinuationToken'] = token
        r = s3.list_objects_v2(**kw)
        for o in r.get('Contents', []):
            m = re.match(r'tile_(\d+)_(\d+)\.bin$', o['Key'])
            if m: done.add((int(m.group(1)), int(m.group(2))))
        if r.get('IsTruncated'): token = r['NextContinuationToken']
        else: break
    return done

DONE = list_done()
TODO = [t for t in ALL if t not in DONE]
print(f'schon auf R2: {len(DONE)} | noch zu tun: {len(TODO)}')

In [ ]:
# GeoTIFF -> byte-kompatibles Uint16-cm-Grid (1000x1000, row0=Süden, nodata=0)
# Quelle bDOM ist 0.2 m (5000x5000) -> auf 1 m (1000x1000) heruntergerechnet.
def make_grid(tif_bytes, tx, ty):
    with rasterio.open(io.BytesIO(tif_bytes)) as src:
        dst = np.zeros((GRID, GRID), dtype=np.float32)
        # Ziel-Raster: north-up, NW-Ecke der km-Kachel, 1 m Auflösung
        dst_transform = from_origin(tx * TILE_SIZE, (ty + 1) * TILE_SIZE,
                                    TILE_SIZE / GRID, TILE_SIZE / GRID)
        reproject(
            source=rasterio.band(src, 1), destination=dst,
            src_transform=src.transform, src_crs=src.crs,
            dst_transform=dst_transform, dst_crs=src.crs,
            src_nodata=src.nodata, dst_nodata=0.0,
            resampling=Resampling.max)   # "höchster Punkt gewinnt" wie das alte DSM;
                                         # erhält Hindernis-Höhen (Baum-/Hauskanten) für die Sicht
    dst = np.flipud(dst)                  # GeoTIFF ist north-up -> unser Format: row0=Süden
    dst[~np.isfinite(dst)] = 0.0
    dst[dst < 0] = 0.0
    return (dst * 100.0).astype('<u2')    # cm, little-endian Uint16

def process(tile):
    tx, ty = tile
    url = f'{BDOM_BASE}/bdom_33{tx}-{ty}.zip'
    r = requests.get(url, timeout=300); r.raise_for_status()
    zf = zipfile.ZipFile(io.BytesIO(r.content))
    tif = next(n for n in zf.namelist() if n.lower().endswith(('.tif', '.tiff')))
    raw = make_grid(zf.read(tif), tx, ty).tobytes()
    assert len(raw) == GRID * GRID * 2, f'falsche Größe {len(raw)}'
    s3.put_object(Bucket=BUCKET, Key=f'tile_{tx}_{ty}.bin', Body=raw,
                  ContentType='application/octet-stream')
    if UPLOAD_GZ:
        s3.put_object(Bucket=BUCKET, Key=f'tile_{tx}_{ty}.bin.gz',
                      Body=gzip.compress(raw), ContentType='application/gzip')
    return tile

In [ ]:
# Lauf (re-runnable: bei Disconnect Resume-Zelle + diese Zelle erneut ausführen)
from tqdm.auto import tqdm
errors = []
with ThreadPoolExecutor(max_workers=WORKERS) as ex:
    futs = {ex.submit(process, t): t for t in TODO}
    for f in tqdm(as_completed(futs), total=len(futs)):
        try:
            f.result()
        except Exception as e:
            errors.append((futs[f], str(e)))
print('fertig. Fehler:', len(errors))
for t, e in errors[:10]:
    print(' ', t, e)

In [ ]:
# Plausibilitäts-Check einer Kachel aus R2
tx, ty = (ALL[0] if ALL else (459, 5722))
raw = s3.get_object(Bucket=BUCKET, Key=f'tile_{tx}_{ty}.bin')['Body'].read()
a = np.frombuffer(raw, dtype='<u2').reshape(GRID, GRID) / 100.0
v = a[a > 0]
print(f'tile_{tx}_{ty}: min={v.min():.1f} mean={v.mean():.1f} max={v.max():.1f} m | '
      f'Abdeckung={100*v.size/a.size:.1f}%')